In [ ]:
# SP-7: Action Label Creation (PySpark Version)
# Creates training labels based on actual outcomes

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType
import time
import json

spark = SparkSession.builder.getOrCreate()

print("=" * 80)
print("SP-7: Action Label Creation (PySpark)")
print("=" * 80)
start_time = time.time()

In [ ]:
# Configuration
WORKSPACE_DIR = '/Workspace/Users/leo.lwakabamba@gmail.com/poker-ml-data/'
UC_VOLUME_DIR = '/Volumes/pokerml/default/data/'

# Input: SP-6 output (from UC Volume)
SP6_INPUT_PATH = UC_VOLUME_DIR + 'processed/sp6_features_enhanced'

# Output path (to UC Volume)
OUTPUT_PATH = UC_VOLUME_DIR + 'processed/sp7_action_labels'

print(f"Input: {SP6_INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")

In [ ]:
# Load data
print("\n[1/5] Loading enhanced features...")

# HARDCODED PATH
df = spark.read.parquet('/Volumes/pokerml/default/data/processed/sp6_features_enhanced')
initial_count = df.count()
print(f"   Loaded {initial_count:,} actions")
print(f"   Columns: {len(df.columns)}")

# ============================================================================
# TRACE PATTERN: Select a hand_id that has ALL STREETS for proper validation
# ============================================================================
print("\n[DEBUG] Finding a hand with all streets for tracing...")

# Count streets per hand
hand_streets = df.groupBy('hand_id').agg(
    F.countDistinct('street').alias('num_streets'),
    F.count('*').alias('num_actions')
)

# Find hands with all 4 streets (preflop, flop, turn, river)
full_hands = hand_streets.filter(F.col('num_streets') >= 4).orderBy(F.col('num_actions').desc())
full_hand_count = full_hands.count()
print(f"   Hands with all 4 streets: {full_hand_count}")

if full_hand_count > 0:
    TRACE_HAND_ID = full_hands.first()['hand_id']
    print(f"   Selected TRACE_HAND_ID with all streets: {TRACE_HAND_ID}")
else:
    # Fallback to hand with most streets
    best_hand = hand_streets.orderBy(F.col('num_streets').desc(), F.col('num_actions').desc()).first()
    TRACE_HAND_ID = best_hand['hand_id']
    print(f"   WARNING: No hands with all 4 streets found!")
    print(f"   Selected TRACE_HAND_ID with {best_hand['num_streets']} streets: {TRACE_HAND_ID}")

# Check street distribution
print(f"\n[DEBUG] Street distribution:")
df.groupBy('street').count().orderBy('street').show()

# Check action_type distribution
print(f"\n[DEBUG] Action type distribution:")
df.groupBy('action_type').count().orderBy('count', ascending=False).show()

# TRACE: Show all actions for selected hand
print(f"\n[TRACE] RAW INPUT for {TRACE_HAND_ID} (should show all streets):")
trace_cols = ['hand_id', 'idx', 'actor', 'street', 'action_type', 'amount', 'facing_call', 'pot_before_action']
trace_cols = [c for c in trace_cols if c in df.columns]
df.filter(F.col('hand_id') == TRACE_HAND_ID).select(trace_cols).orderBy('idx').show(30, truncate=False)

# Verify this hand has multiple streets
trace_streets = df.filter(F.col('hand_id') == TRACE_HAND_ID).select('street').distinct().collect()
print(f"   Streets in traced hand: {[r['street'] for r in trace_streets]}")

# Show key columns available
print(f"\n   Key columns available:")
key_cols = ['hand_equity', 'target_profit_bb', 'spr', 'pot_odds_favorable', 'position_late', 'hole_cards', 'action_idx', 'idx']
for col in key_cols:
    exists = col in df.columns
    print(f"      {col}: {'✓' if exists else 'MISSING'}")

In [ ]:
# Ensure required columns exist
print("\n[2/5] Preparing columns...")

# Add defaults for missing columns
if 'amount' not in df.columns:
    df = df.withColumn('amount', F.lit(0.0))
if 'facing_call' not in df.columns:
    df = df.withColumn('facing_call', F.lit(0.0))
if 'pot_before_action' not in df.columns:
    df = df.withColumn('pot_before_action', F.lit(1.0))
if 'starting_stack' not in df.columns:
    df = df.withColumn('starting_stack', F.lit(100.0))
if 'target_profit_bb' not in df.columns:
    df = df.withColumn('target_profit_bb', F.lit(0.0))
if 'hand_equity' not in df.columns:
    df = df.withColumn('hand_equity', F.lit(0.5))
if 'spr' not in df.columns:
    df = df.withColumn('spr', F.lit(10.0))
if 'pot_odds_favorable' not in df.columns:
    df = df.withColumn('pot_odds_favorable', F.lit(0))
if 'position_late' not in df.columns:
    df = df.withColumn('position_late', F.lit(0))

# Calculate intermediate columns for action mapping
df = df.withColumn('bet_size', F.col('amount') - F.col('facing_call'))
df = df.withColumn(
    'pot_fraction',
    F.when(F.col('pot_before_action') > 0,
           F.col('bet_size') / F.col('pot_before_action'))
    .otherwise(0.0)
)
df = df.withColumn(
    'stack_fraction',
    F.col('amount') / (F.col('starting_stack') + 0.000001)
)

print("   Columns prepared")

In [ ]:
# Map actions to categories
print("\n[3/5] Mapping actual actions to policy categories...")

# Map action_type to actual_action_category using nested when/otherwise
df = df.withColumn(
    'actual_action_category',
    F.when(F.col('action_type') == 'fold', 'fold')
    .when(F.col('action_type').isin(['call_or_check', 'call', 'check']), 'call')
    .when(
        F.col('action_type').isin(['bet_or_raise_to', 'raise', 'bet']) & (F.col('stack_fraction') >= 0.9),
        'shove'
    )
    .when(
        F.col('action_type').isin(['bet_or_raise_to', 'raise', 'bet']) & (F.col('pot_fraction') >= 1.0),
        'shove'
    )
    .when(
        F.col('action_type').isin(['bet_or_raise_to', 'raise', 'bet']) & (F.col('pot_fraction') >= 0.5),
        'raise_75'
    )
    .when(
        F.col('action_type').isin(['bet_or_raise_to', 'raise', 'bet']) & (F.col('pot_fraction') < 0.5),
        'raise_33'
    )
    .otherwise('call')
)

# Show action distribution
print("\n   Actual action distribution:")
action_dist = df.groupBy('actual_action_category').count().orderBy('count', ascending=False)
action_dist.show()

# TRACE: After action mapping - show hand, bet, street for debugging
print(f"\n[TRACE] After action mapping for {TRACE_HAND_ID}:")
print("   Raw hand/bet/street columns:")
trace_cols = ['hand_id', 'actor', 'street', 'action_type', 'amount']
# Add card columns if they exist
for col in ['hole_cards', 'cards', 'board', 'flop', 'turn', 'river', 'community_cards']:
    if col in df.columns:
        trace_cols.append(col)
# Add bet-related columns
for col in ['facing_call', 'pot_before_action', 'starting_stack', 'bb']:
    if col in df.columns:
        trace_cols.append(col)
trace_cols.append('actual_action_category')

df.filter(F.col('hand_id') == TRACE_HAND_ID).select(trace_cols).show(20, truncate=False)

In [ ]:
# Create labels based on profitability
print("\n[4/5] Creating labels from actual outcomes...")

# Determine best_action using heuristics when not profitable
# Priority: If profitable, keep actual action; otherwise use strategy heuristics
# NOTE: hand_equity = TRUE hand strength from showdown (0-1 scale)

df = df.withColumn(
    'best_action',
    # Very profitable (>2 BB) - keep actual
    F.when(F.col('target_profit_bb') > 2.0, F.col('actual_action_category'))
    # Profitable (>0.5 BB) - keep actual
    .when(F.col('target_profit_bb') > 0.5, F.col('actual_action_category'))
    # Weak hand, bad position, no pot odds -> fold
    .when(
        (F.col('hand_equity') < 0.3) & 
        (F.col('pot_odds_favorable') == 0) & 
        (F.col('position_late') == 0),
        'fold'
    )
    # Strong hand, low SPR -> shove
    .when(
        (F.col('hand_equity') >= 0.7) & (F.col('spr') < 3),
        'shove'
    )
    # Strong hand, medium SPR -> raise_75
    .when(
        (F.col('hand_equity') >= 0.7) & (F.col('spr') < 7),
        'raise_75'
    )
    # Strong hand, deep -> raise_33
    .when(
        F.col('hand_equity') >= 0.7,
        'raise_33'
    )
    # Medium hand, late position -> raise_33
    .when(
        (F.col('hand_equity') >= 0.4) & (F.col('position_late') == 1),
        'raise_33'
    )
    # Drawing hand with odds -> call
    .when(
        (F.col('hand_equity') >= 0.3) & 
        ((F.col('pot_odds_favorable') == 1) | (F.col('facing_call') < F.col('pot_before_action') * 0.5)),
        'call'
    )
    # Default -> fold
    .otherwise('fold')
)

# Override: if actual action was profitable, always keep it
df = df.withColumn(
    'best_action',
    F.when(F.col('target_profit_bb') > 0.5, F.col('actual_action_category'))
    .otherwise(F.col('best_action'))
)

print(f"   Created labels")

# Show best_action distribution
print("\n   Best action distribution:")
df.groupBy('best_action').count().orderBy('count', ascending=False).show()

# TRACE: After label creation
print(f"\n[TRACE] After label creation for {TRACE_HAND_ID}:")
df.filter(F.col('hand_id') == TRACE_HAND_ID).select(
    'hand_id', 'actor', 'street', 'action_type',
    'actual_action_category', 'best_action',
    'target_profit_bb', 'hand_equity', 'spr'
).show(20, truncate=False)

In [ ]:
# Analyze label distribution by street
print("\n   Action distribution by street:")

for street in ['preflop', 'flop', 'turn', 'river']:
    street_df = df.filter(F.col('street') == street)
    street_count = street_df.count()
    if street_count == 0:
        continue
    print(f"\n   {street.upper()} ({street_count:,} actions):")
    street_df.groupBy('best_action').count().orderBy('count', ascending=False).show(truncate=False)

In [ ]:
# Add dummy EV columns for compatibility and clean up
print("\n[5/5] Saving labeled dataset...")

# Add EV columns
df = df.withColumn('fold_ev', F.lit(0.0))
df = df.withColumn('call_ev', F.lit(0.0))
df = df.withColumn('raise_33_ev', F.lit(0.0))
df = df.withColumn('raise_75_ev', F.lit(0.0))
df = df.withColumn('shove_ev', F.lit(0.0))

# Drop temporary columns
drop_cols = ['bet_size', 'pot_fraction', 'stack_fraction']
df = df.drop(*[c for c in drop_cols if c in df.columns])

# TRACE: Final output before save
print(f"\n[TRACE] FINAL OUTPUT for {TRACE_HAND_ID}:")
df.filter(F.col('hand_id') == TRACE_HAND_ID).select(
    'hand_id', 'actor', 'street', 'action_type',
    'actual_action_category', 'best_action',
    'target_profit_bb', 'hand_equity'
).show(20, truncate=False)

# Save as Parquet - HARDCODED PATH
df.write.mode('overwrite').parquet('/Volumes/pokerml/default/data/processed/sp7_action_labels')

# Get final stats
final_count = df.count()
final_cols = len(df.columns)

elapsed = time.time() - start_time

print("\n" + "=" * 80)
print("SP-7 Action Label Creation COMPLETE (PySpark)!")
print("=" * 80)
print(f"\nRuntime: {elapsed:.1f} seconds")
print(f"Dataset: {final_count:,} actions with labels, {final_cols} columns")
print(f"\nData Lineage:")
print(f"  Input: SP-6 output (/Volumes/pokerml/default/data/processed/sp6_features_enhanced)")
print(f"  Output: /Volumes/pokerml/default/data/processed/sp7_action_labels")
print(f"\n[TRACE] TRACE_HAND_ID used: {TRACE_HAND_ID}")
print(f"\nNext step: Run SP-8 (08_PolicyTraining) to train policy classifiers")

In [ ]:
# Show sample of labeled data
print("\nSample of labeled data:")
df.select(
    'hand_id', 'street', 'action_type',
    'actual_action_category', 'best_action',
    'target_profit_bb', 'hand_equity'
).show(15, truncate=False)

---
## OLD PANDAS VERSION (Commented Out)
The code below is the original pandas-based implementation.
Kept for reference.

In [ ]:
# # OLD VERSION - PANDAS IMPLEMENTATION
# # ====================================
# 
# # SP-7: Action Label Creation (Learn from Actual Play)
# # Creates training labels based on actual outcomes
# 
# import pandas as pd
# import numpy as np
# import json
# import time
# from pathlib import Path
# 
# # ... (rest of original pandas implementation)
# # See git history for full original code